In [ ]:
import sisl
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

First Siesta example for analysing output.

This example will calculate the electronic structure of graphene using Siesta, and we will post-process the data to calculate band-structures etc.

We remark that settings during the Siesta/TranSiesta tutorials are not necessarily converged or proper settings. Users need to invest time in understanding the intrinsics of Siesta before performing real calculations!

## Exercises

As this is your first example of running Siesta, there are a few things you should know:

1. Start by calculating the electronic structure using Siesta

       siesta RUN.fdf > RUN.out

  - Go through the output of Siesta to get used to it, also to assert that it ran without errors (always do this!) ((always, **always** do THIS!))
  - Ensure that the self-consistent field (SCF) has indeed converged, Siesta will, by default, die if it hasn't converged the SCF.
  
2. Use `sisl` to read in the electronic structure (the Hamiltonian). There are several ways of doing this, for now you should try these two methods:

         H_fdf = sisl.Hamiltonian.read('RUN.fdf')
         H_HSX = sisl.Hamiltonian.read('siesta.HSX')
         
    Print the objects and note the difference between the two objects.
    What do you think these differences mean? (it isn't relevant for this tutorial, but it will be in later tutorials).
    
3. Calculate the $\Gamma$-point eigen-spectrum using both above Hamiltonians (search the `sisl` documentation for `eigh`). Are they different? If so, why, if not, why not?

4. Calculate the band-structure from the DFT electronic structure using `sisl`. Also calculate the band-structure using the tight-binding Hamiltonian ([TB 1](../TB_01/TB_01.ipynb)) and compare the two.  
*HINT*: Zoom in on an energy-range from $-3$ eV to $3$ eV and plot both the DFT bands and the TB bands in the same plot.  

In [ ]:
%%bash
#%solution clear
siesta RUN.fdf > RUN.out

In [ ]:
# Read Hamiltonians using two different methods
# %solution start
H_fdf = sisl.Hamiltonian.read("RUN.fdf")
H_HSX = sisl.Hamiltonian.read("siesta.HSX")
print(H_fdf)
print(H_HSX)

In [ ]:
# Calculate eigenspectrum at the \Gamma-point
eig_fdf = H_fdf.<>
eig_HSX = H_HSX.<>

In [ ]:
# %solution clear
eig_fdf = H_fdf.eigenstate()
eig_HSX = H_HSX.eigenstate()
print("Check if they are the same:")
print(eig_fdf.eig - eig_HSX.eig)

In [ ]:
# Calculate band-structure from DFT Hamiltonian and TB Hamiltonian
H_tb = sisl.Hamiltonian(sisl.geom.graphene())
H_tb.construct([(0.1, 1.45), (0, -2.7)])

band = sisl.BandStructure(
    H_tb,
    [[0.0, 0.0], [2.0 / 3, 1.0 / 3], [1.0 / 2, 1.0 / 2], [1.0, 1.0]],
    301,
    [r"$\Gamma$", "K", "M", r"$\Gamma$"],
)
xtick, xtick_label = band.lineartick()
lk = band.lineark()

ax = plt.gca()
ax.xaxis.set_ticks(xtick)
ax.set_xticklabels(xtick_label)

# Define limits of the y-axis (Energy!)
# Play with this if you want! :)
ax.set_ylim(-3, 3)
ax.set_ylabel("Eigenspectrum [eV]")

# Plot x-major lines at the ticks
ymin, ymax = ax.get_ylim()
for tick in xtick:
    ax.plot([tick, tick], [ymin, ymax], "k")

# You need to create the TB Hamiltonian, see e.g. TB_01
eigs = band.apply.ndarray.eigh()
ax.plot(lk, eigs, "--")

# Plot band-structures
band.set_parent(H_fdf)
eigs = band.apply.ndarray.eigh()
ax.plot(lk, eigs);

In [ ]:
# Calculate band-structure from DFT Hamiltonian and TB Hamiltonian
band = sisl.BandStructure(sisl.geom.graphene(), <fill-in correct points and labels>)
xtick, xtick_label = band.lineartick()
lk = band.lineark()

ax = plt.gca()
ax.xaxis.set_ticks(xtick)
ax.set_xticklabels(xtick_label)

# Define limits of the y-axis (Energy!)
# Play with this if you want! :)
ax.set_ylim(-3, 3)
ax.set_ylabel('Eigenspectrum [eV]')

# Plot x-major lines at the ticks
ymin, ymax = ax.get_ylim()
for tick in xtick:
    ax.plot([tick,tick], [ymin, ymax], 'k')

# Plot band-structures
band.set_parent(<DFT Hamiltonian>)
eigs = band.apply.ndarray.eigh()
ax.plot(lk, eigs)
    
# You need to create the TB Hamiltonian, see e.g. TB_01
band.set_parent(<TB Hamiltonian>)
eigs = band.apply.ndarray.eigh()
ax.plot(lk, eigs, '--')

Continue with plotting a fatbands plot using [`sisl.viz`](https://zerothi.github.io/sisl/visualization/showcase/FatbandsPlot.html).
- There is only one species, so figure out how to plot individual orbitals, and/or how to expand the plot to do fatband analysis on all orbitals.
  
In this exercise you can re-use the above `band` object. Since the eigenspectrum is *large*, you should limit the $y$-range in the plot. Use `update_layout(yaxis_range=[<min>, <max>])` on the `plotly` object.